In [ ]:

import os, sys, glob, json, time, re, shutil, subprocess
def mount(owner, slug, required):
    for p in (f"/kaggle/input/datasets/{owner}/{slug}", f"/kaggle/input/{slug}"):
        if os.path.exists(os.path.join(p, required)):
            return p
    raise FileNotFoundError(f"{owner}/{slug}/{required} not mounted")
COMP = next(p for p in ("/kaggle/input/competitions/rsna-knee-abnormality-detection",
                        "/kaggle/input/rsna-knee-abnormality-detection") if os.path.exists(p + "/train.csv"))
print("competition:", COMP)

import numpy as np, pandas as pd, cv2
from multiprocessing import Pool
from sklearn.model_selection import GroupKFold

GP = mount("goodpjw2008", "rsna-knee-2-5d-convnext-reader", "preprocess.py")
for w in sorted(glob.glob(GP + "/*.whl")):
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "-q", w], check=False)
SRC = "/kaggle/working/src"
os.makedirs(SRC, exist_ok=True)
for f in ("preprocess.py", "knee.py"):
    shutil.copy(f"{GP}/{f}", f"{SRC}/{f}")
sys.path.insert(0, SRC)
import preprocess
preprocess.MAX_SLICES = 32

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
PLANES = ["Sagittal", "Coronal", "Axial"]
RES = 256
OUTC = "/kaggle/working/cache256"
os.makedirs(OUTC, exist_ok=True)

se = pd.read_csv(COMP + "/train_series.csv")
print(se.columns.tolist(), len(se))

def pick(study_df, root):
    """slot -> series id: the series with the most files in that slot (stable on ties)."""
    best = {}
    for sr, fs, pl in zip(study_df.SeriesInstanceUID, study_df.Fat_Suppression, study_df.Anatomical_Plane):
        if pl not in PLANES:
            continue
        slot = PLANES.index(pl) + 3 * (1 - int(fs))
        d = f"{root}/{study_df.StudyInstanceUID.iloc[0]}/{sr}"
        n = len(os.listdir(d)) if os.path.isdir(d) else 0
        if n >= 3 and (slot not in best or n > best[slot][1]):
            best[slot] = (str(sr), n)
    return {s: v[0] for s, v in best.items()}

ROOT = COMP + "/train_series"
groups = [(st, g) for st, g in se.groupby("StudyInstanceUID")]

def job(item):
    st, g = item
    out = f"{OUTC}/{st}.npz"
    try:
        sel = pick(g, ROOT)
        arrs = {}
        for slot, sr in sel.items():
            plane = PLANES[slot % 3]
            vol, info = preprocess.load_series(f"{ROOT}/{st}/{sr}", plane)
            if len(vol) < 3:
                continue
            # JPEG q95 per slice (raw uint8 overflowed the 20 GB output disk): concatenated bytes + offsets
            blobs = [cv2.imencode(".jpg", cv2.resize(v, (RES, RES), interpolation=cv2.INTER_AREA),
                                  [cv2.IMWRITE_JPEG_QUALITY, 95])[1].tobytes() for v in vol]
            arrs[f"s{slot}_d"] = np.frombuffer(b"".join(blobs), np.uint8)
            arrs[f"s{slot}_o"] = np.cumsum([0] + [len(b) for b in blobs]).astype(np.int64)
        np.savez(out, **arrs)
        return st, len(arrs) // 2, ""
    except Exception as e:
        return st, 0, f"{type(e).__name__}: {e}"[:200]

t0 = time.time()
rows = []
with Pool(4) as p:
    for i, r in enumerate(p.imap_unordered(job, groups, chunksize=2)):
        rows.append(r)
        if (i + 1) % 200 == 0:
            print(f"{i+1}/{len(groups)} {time.time()-t0:.0f}s", flush=True)
meta = pd.DataFrame(rows, columns=["StudyInstanceUID", "n_slots", "error"])
meta.to_csv("/kaggle/working/cache256_meta.csv", index=False)
print("cache done", len(meta), "errors", int((meta.error != "").sum()), "zero-slot", int((meta.n_slots == 0).sum()),
      f"{time.time()-t0:.0f}s")
print(subprocess.run(["du", "-sh", OUTC], capture_output=True, text=True).stdout)

# ---- labels ----
KJ = mount("karttikjangid05", "rsna-knee-public-llm-label-sets-compared", "consensus_labels.csv")
cons = pd.read_csv(KJ + "/consensus_labels.csv").set_index("StudyInstanceUID")[LABELS]
tr = pd.read_csv(COMP + "/train.csv")
ids = tr.StudyInstanceUID
gold = tr.set_index("StudyInstanceUID")[LABELS]
is_gold = gold.notna().all(axis=1)
soft = cons.reindex(ids).astype(float)
print("missing consensus rows:", int(soft.isna().any(axis=1).sum()))
soft = soft.fillna(soft.mean()).clip(0.01, 0.99)
out = soft.copy()
out.insert(0, "is_gold", is_gold.astype(int).values)
out["fold"] = -1
weak = ~is_gold.values
grp = tr.Report.map(lambda t: re.sub(r"\W+", "", str(t).lower())).values[weak]
order = np.random.RandomState(0).permutation(weak.sum())
folds = np.zeros(weak.sum(), int)
for k, (_, va) in enumerate(GroupKFold(5).split(order, groups=grp[order])):
    folds[order[va]] = k
out.loc[weak, "fold"] = folds
for c in LABELS:
    out[f"gold_{c}"] = gold[c].values
out.to_csv("/kaggle/working/labels_cons.csv")
print("labels", out.shape, out.fold.value_counts().sort_index().to_dict())
from sklearn.metrics import roc_auc_score
g = out[out.is_gold == 1]
print("consensus vs gold58 macro AUC:", np.mean([roc_auc_score(g[f"gold_{c}"], g[c]) for c in LABELS]).round(4))
